# Hillsborough County Parcel Comp Analysis: Exploratory Data Analysis

Setup the workspace

In [ ]:
import geopandas as gpd
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

Read in parquet files and drop duplicate columns by prioritizing the sales dataset.

In [ ]:
df_sales = pd.read_parquet('../data/processed/allsales.parquet')
gdf_parcel = gpd.read_parquet('../data/processed/parcel.parquet')

In [ ]:
gdf = gdf_parcel.merge(df_sales, on='FOLIO', how='right')
gdf.drop([col for col in gdf.columns if col.endswith('_x')], axis=1, inplace=True)
gdf.rename(columns={col: col[:-2] for col in gdf.columns if col.endswith('_y')}, inplace=True)

Explore the sales data. Refer to the [allsales](../docs/allsales.pdf) document provided by Hillsborough County Property Appraiser for a description about these fields.

In [ ]:
gdf.dtypes

There are 63 fields available for analysis; however, many of these are either correlated such as address, city and zip or not suitable for modeling such as legal descriptions. Let's reduce the width to just the key fields a potential buyer might be aware of while preserving a primary key to rejoin should we need to in the future and make sure the data types are appropriate. Since the size of the data is sufficient imputing missing values isn't requiered and we can drop the records entirely.

In [ ]:
gdf_reduced = gdf[['FOLIO', 'SITE_ZIP', 'tBEDS', 'tBATHS', 
                   'tSTORIES', 'tUNITS', 'tBLDGS', 'HEAT_AR', 
                   'ACREAGE', 'LU_GRP', 'S_AMT', 'S_YEAR']].copy().dropna()

gdf_reduced['SITE_ZIP'] = (gdf_reduced['SITE_ZIP'].astype(str).str.split('-').str[0].str.zfill(5))
gdf_reduced['tBEDS'] = gdf_reduced['tBEDS'].astype(float)
gdf_reduced['tBATHS'] = gdf_reduced['tBATHS'].astype(float)
gdf_reduced['tSTORIES'] = gdf_reduced['tSTORIES'].astype(float)
gdf_reduced['tUNITS'] = gdf_reduced['tUNITS'].astype(float)
gdf_reduced['tBLDGS'] = gdf_reduced['tBLDGS'].astype(float)
gdf_reduced['HEAT_AR'] = gdf_reduced['HEAT_AR'].astype(float)
gdf_reduced['ACREAGE'] = gdf_reduced['ACREAGE'].astype(float)
gdf_reduced['S_AMT'] = gdf_reduced['S_AMT'].astype(float)
gdf_reduced['S_YEAR'] = gdf_reduced['S_YEAR'].astype('category')
gdf_reduced['LU_GRP'] = gdf_reduced['LU_GRP'].astype('category')
gdf_reduced['FOLIO'] = gdf_reduced['FOLIO'].astype(str)
gdf_reduced['SITE_ZIP'] = gdf_reduced['SITE_ZIP'].astype('category')

In [ ]:
print('Original shape', gdf.shape, '\n'
      'Reduced shape', gdf_reduced.shape)

In [ ]:
gdf_reduced.dtypes

In [ ]:
# plot the number of sales per year
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Ensure years are sorted numerically
years = sorted(gdf_reduced['S_YEAR'].unique())
sns.countplot(data=gdf_reduced, x='S_YEAR', order=years)

# Filter ticks to show every 10 years
tick_years = [year for year in years if year % 10 == 0]
plt.xticks(ticks=[years.index(y) for y in tick_years], labels=tick_years, rotation=45)

plt.title('Number of Sales per Year')
plt.xlabel('Year')
plt.ylabel('Number of Sales')
plt.tight_layout()

Check for correllation and multicollinearity for the numeric fields

In [ ]:
numeric_cols = gdf_reduced.select_dtypes(include=['number'])
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt='.2f')

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

vif_const = add_constant(numeric_cols)
# Calculate VIF
vif_df = pd.DataFrame({
    'feature': vif_const.columns,
    'VIF': [variance_inflation_factor(vif_const.values, i) for i in range(vif_const.shape[1])]
})

vif_df.sort_values(by='VIF', ascending=False)

There is a strong correlation between beds to baths and stories to buildings, and all predictors have a weak correlation between sales price. To make sure multicollinearity isn't a concern, a Variance Inflation Factor (VIF) analysis shows that there is no high multicollinearity (>10) with moderate levels for stories and beds so we can keep these predictors in our analysis.

Take a look at the count of records by the generalized land use group.

In [ ]:
pd.DataFrame({
    'Count': gdf_reduced['LU_GRP'].value_counts(),
    'Percentage': round(gdf_reduced['LU_GRP'].value_counts(normalize=True) * 100,2)
})

There are 37 land use groups with the majority of property sales being residential consisting of single family (77%) and multi-family (10%) properties. Let's focus on just residential properties since that's where the activity is and will yeild better results when modeling.

In [ ]:
gdf_resid = gdf_reduced.loc[gdf_reduced['LU_GRP'].isin(['SINGLE FAMILY', 'MULTI-FAMILY'])]

Show the distribution of sale amount by land use group using violin plots.

In [ ]:
# Filter to top categories
# top_groups = gdf['LU_GRP'].value_counts().nlargest(5).index
grouped = {grp: gdf_resid[gdf_resid['LU_GRP'] == grp] for grp in gdf_resid['LU_GRP'].unique()}

# Create traces
fig = go.Figure()

for i, (grp, df_grp) in enumerate(grouped.items()):
    fig.add_trace(go.Violin(
        y=df_grp['S_AMT'],
        name=grp,
        visible=(i == 0),
        box_visible=True,
        meanline_visible=True
    ))

# Dropdown menu
buttons = [
    dict(label=grp,
         method="update",
         args=[{"visible": [i == j for j in range(len(grouped))]},
               {"title": f"Sales Amount for {grp}"}])
    for i, grp in enumerate(grouped.keys())
]

fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.05,
        y=1,
    )],
    title="Sales Amount by Land Use Group",
    yaxis_title="S_AMT"
)

fig.show()

This shows there are not only outliers in the data due to manual data entry, but there might also be sentinel values or people gifting homes to family any avoiding taxes represented by the high quantity of single family and multi-family homes recorded at a sales price of $100. Let's try to clean the data by removing sentinel values, only keep the last 10 years of data, and then remove outliers. 

Find the best cutoff value for a realistic sales price

In [ ]:
# plot a histogram of the sales amount
sns.histplot(gdf_resid['S_AMT'], bins=200, kde=True)

In [ ]:
# Remove sales <= 50k and keep only last 10 years of data
gdf_resid_nosents = gdf_resid[(gdf_resid['S_YEAR'].astype(int).between(2014, 2024)) & (gdf_resid['S_AMT'] > 50000)]

# plot a histogram of the sales amount
sns.histplot(gdf_resid_nosents['S_AMT'], bins=200, kde=True)

Due to the size of the dataframe and the non-normal distribution, outlier detection methods such as using Cooks' distance or IQR won't perform well, but there are machine learning available for large, skewed datasets - for example: Isolation Forest.

In [ ]:
from sklearn.ensemble import IsolationForest


# Make a copy to avoid modifying original
gdf_labeled = gdf_resid_nosents.copy()
gdf_labeled['anomaly'] = None

# Loop over each year group
for year, group_idx in gdf_labeled.groupby('S_YEAR').groups.items():
    group = gdf_labeled.loc[group_idx]
    numeric_group = group.select_dtypes(include=['number'])

    # Skip if not enough data due to Isolation Forest requirements
    if len(numeric_group) < 2:
        continue

    # Fit Isolation Forest
    iso = IsolationForest(n_estimators=250, contamination=0.025, random_state=123)
    preds = iso.fit_predict(numeric_group)

    # Assign back to main dataframe
    gdf_labeled.loc[group_idx, 'anomaly'] = preds

# plotting the results
fig, axes = plt.subplots(3, 1, figsize=(10, 14), sharey=True)

# All points plot
sns.scatterplot(
    data=gdf_labeled,
    x='HEAT_AR', y='S_AMT',
    color='gray',
    ax=axes[0]
)
axes[0].set_title(f'Price vs. Acreage (All Points) n={len(gdf_labeled)}')

# Outliers plot
sns.scatterplot(
    data=gdf_labeled[gdf_labeled['anomaly'] == -1],
    x='HEAT_AR', y='S_AMT',
    color='red',
    ax=axes[1]
)
axes[1].set_title(f'Price vs. Acreage (Outliers) n={len(gdf_labeled[gdf_labeled["anomaly"] == -1])}')

# Normal points plot
sns.scatterplot(
    data=gdf_labeled[gdf_labeled['anomaly'] == 1],
    x='HEAT_AR', y='S_AMT',
    color='blue',
    ax=axes[2]
)
axes[2].set_title(f'Price vs. Acreage (Inliers) n={len(gdf_labeled[gdf_labeled["anomaly"] == 1])}')



plt.tight_layout()
plt.show()

These plots align with the concept that most houses being sold in a city have lower acreage. The higher the acreage, the more likely it is in rural areas where the home value is less than in urban areas.

Remove the outliers from the dataframe

In [ ]:
df_clean = gdf_labeled[gdf_labeled['anomaly'] == 1].drop(columns=['anomaly'], axis=1).copy()
df_clean.shape

Replot the data without outliers

In [ ]:
grouped = {grp: df_clean[df_clean['LU_GRP'] == grp] for grp in df_clean['LU_GRP'].unique()}

# Create traces
fig = go.Figure()

for i, (grp, df_grp) in enumerate(grouped.items()):
    fig.add_trace(go.Violin(
        y=df_grp['S_AMT'],
        name=grp,
        visible=(i == 0),
        box_visible=True,
        meanline_visible=True
    ))

# Dropdown menu
buttons = [
    dict(label=grp,
         method="update",
         args=[{"visible": [i == j for j in range(len(grouped))]},
               {"title": f"Sales Amount for {grp}"}])
    for i, grp in enumerate(grouped.keys())
]

fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.05,
        y=1,
    )],
    title="Sales Amount by Land Use Group",
    yaxis_title="S_AMT"
)

fig.show()

In [ ]:
import plotly.express as px

# Create boxplot
fig = px.box(
    df_clean,
    x='S_YEAR',
    y='S_AMT',
    color='LU_GRP',
    title='Sales Amount by Year and Land Use Group',
    labels={'S_AMT': 'Sales Amount', 'S_YEAR': 'Year', 'LU_GRP': 'Land Use Group'},
    points='outliers'  # set to 'all' or 'outliers' if you want to show points
)

fig.update_layout(
    boxmode='group',  # group boxes side-by-side by LU_GRP
    xaxis_type='category'
)

fig.show()

Export the cleaned data for model training

In [ ]:
# eport the cleaned data
df_clean.to_parquet('../data/processed/cleaned.parquet', index=False)